# W11-D6 概念实验：实施路线图依赖 DAG

核心概念来自 Markdown：路线图按风险与依赖排序，Sprint 0 先清理术语，随后补 KnowledgeSnapshot、RuntimeABI + OCI、CapabilityRelease，Connector 统一抽象放到后续。下面用标准库做 DAG 拓扑排序。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False


In [ ]:
from collections import defaultdict, deque

nodes = ["Sprint 0 / 术语+基线", "Sprint 1 / KnowledgeSnapshot", "Sprint 2 / RuntimeABI+OCI", "Sprint 3 / CapabilityRelease", "Sprint 4+ / Connector"]
edges = [
    (nodes[0], nodes[1]), (nodes[0], nodes[2]),
    (nodes[1], nodes[2]), (nodes[1], nodes[3]),
    (nodes[2], nodes[3]), (nodes[3], nodes[4]),
]
graph = defaultdict(list)
indegree = {node: 0 for node in nodes}
for before, after in edges:
    graph[before].append(after)
    indegree[after] += 1

queue = deque(node for node in nodes if indegree[node] == 0)
order = []
while queue:
    current = queue.popleft()
    order.append(current)
    for nxt in graph[current]:
        indegree[nxt] -= 1
        if indegree[nxt] == 0:
            queue.append(nxt)
assert len(order) == len(nodes), "路线图存在循环依赖"
print("拓扑排序结果:")
for i, item in enumerate(order, 1):
    print(i, item.replace("\n", " / "))

In [ ]:
positions = {nodes[0]: (0, 0), nodes[1]: (1, 1), nodes[2]: (2, 1), nodes[3]: (3, 1), nodes[4]: (4, 0)}
fig, ax = plt.subplots(figsize=(11, 3.5))
for before, after in edges:
    x1, y1 = positions[before]
    x2, y2 = positions[after]
    ax.annotate("", xy=(x2 - 0.12, y2), xytext=(x1 + 0.12, y1), arrowprops={"arrowstyle": "->", "lw": 1.5, "color": "#777"})
for i, node in enumerate(nodes):
    x, y = positions[node]
    color = ["#4c78a8", "#e45756", "#f2cf5b", "#59a14f", "#9c755f"][i]
    ax.scatter([x], [y], s=1700, color=color, alpha=0.9, zorder=2)
    ax.text(x, y, node, ha="center", va="center", fontsize=9, zorder=3)
ax.set_xlim(-0.6, 4.6)
ax.set_ylim(-0.6, 1.6)
ax.axis("off")
ax.set_title("LangChat v2 实施路线图：依赖关系 DAG")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
risk = {nodes[1]: "闭包可复现性", nodes[2]: "版本兼容", nodes[3]: "能力发布治理", nodes[4]: "外部连接治理"}
print("排序不是按客户可见功能，而是按先决依赖和运行时风险：")
for item in order:
    print(f"{item.replace(chr(10), ' / ')} -> {risk.get(item, '术语与基线')}")
print("结论：DAG 把“先修地基再盖楼”变成可检查的依赖约束，任何跳过前置节点的排期都应被标记为风险。")